In [1]:
import ast
from pathlib import Path

In [2]:
file = Path("exceptions.py")

In [3]:
with open(file, encoding="utf-8") as f:
    content= f.read()

tree = ast.parse(content)
print(ast.dump(tree, indent=2))

Module(
  body=[
    Expr(
      value=Constant(value="\nrequests.exceptions\n~~~~~~~~~~~~~~~~~~~\n\nThis module contains the set of Requests' exceptions.\n")),
    ImportFrom(
      module='__future__',
      names=[
        alias(name='annotations')],
      level=0),
    ImportFrom(
      module='typing',
      names=[
        alias(name='TYPE_CHECKING'),
        alias(name='Any')],
      level=0),
    ImportFrom(
      module='urllib3.exceptions',
      names=[
        alias(name='HTTPError', asname='BaseHTTPError')],
      level=0),
    ImportFrom(
      module='compat',
      names=[
        alias(name='JSONDecodeError', asname='CompatJSONDecodeError')],
      level=1),
    If(
      test=Name(id='TYPE_CHECKING', ctx=Load()),
      body=[
        ImportFrom(
          module='models',
          names=[
            alias(name='PreparedRequest'),
            alias(name='Request'),
            alias(name='Response')],
          level=1)],
      orelse=[]),
    ClassDef(
      name='R

In [4]:
for node in tree.body:
    print(type(node).__name__, getattr(node, "name", None), getattr(node, "lineno", None), getattr(node, "end_lineno", None))

Expr None 1 6
ImportFrom None 8 8
ImportFrom None 10 10
ImportFrom None 12 12
ImportFrom None 14 14
If None 16 17
ClassDef RequestException 20 35
ClassDef InvalidJSONError 38 39
ClassDef JSONDecodeError 42 63
ClassDef HTTPError 66 67
ClassDef ConnectionError 70 71
ClassDef ProxyError 74 75
ClassDef SSLError 78 79
ClassDef Timeout 82 88
ClassDef ConnectTimeout 91 95
ClassDef ReadTimeout 98 99
ClassDef URLRequired 102 103
ClassDef TooManyRedirects 106 107
ClassDef MissingSchema 110 111
ClassDef InvalidSchema 114 115
ClassDef InvalidURL 118 119
ClassDef InvalidHeader 122 123
ClassDef InvalidProxyURL 126 127
ClassDef ChunkedEncodingError 130 131
ClassDef ContentDecodingError 134 135
ClassDef StreamConsumedError 138 139
ClassDef RetryError 142 143
ClassDef UnrewindableBodyError 146 147
ClassDef RequestsWarning 153 154
ClassDef FileModeWarning 157 158
ClassDef RequestsDependencyWarning 161 162


In [5]:
def get_start_line(node):
    if hasattr(node, "decorator_list") and node.decorator_list:
        return node.decorator_list[0].lineno
    return node.lineno

In [6]:
for node in tree.body:
    output = get_start_line(node)
    print(output)

1
8
10
12
14
16
20
38
42
66
70
74
78
82
91
98
102
106
110
114
118
122
126
130
134
138
142
146
153
157
161


In [7]:
content_lines = content.splitlines()

def get_text(start_line, end_line):
    return "\n". join(content_lines[start_line -1 : end_line])

In [8]:
node = tree.body[8]
print(get_text(get_start_line(node), node.end_lineno))

class JSONDecodeError(InvalidJSONError, CompatJSONDecodeError):
    """Couldn't decode the text into json"""

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        """
        Construct the JSONDecodeError instance first with all
        args. Then use it's args to construct the IOError so that
        the json specific args aren't used as IOError specific args
        and the error message from JSONDecodeError is preserved.
        """
        CompatJSONDecodeError.__init__(self, *args)
        InvalidJSONError.__init__(self, *self.args, **kwargs)

    def __reduce__(self) -> tuple[Any, ...] | str:
        """
        The __reduce__ method called when pickling the object must
        be the one from the JSONDecodeError (be it json/simplejson)
        as it expects all the arguments for instantiation, not just
        one like the IOError, and the MRO would by default call the
        __reduce__ method from the IOError due to the inheritance order.
        """
        retu

In [9]:
# def class_chunk(node, file_stem):
#     start = get_start_line(node)
#     docstring = ast.get_docstring(node)

#     #non-method body: everything in node.body that ISN'T a method
#     non_method_nodes = [n for n in node.body if not isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]

#     end = node.end_lineno if not any(isinstance(n, (ast.FunctionDef, ast.AsyncFunctinDef)) for n in node.body) else (non_method_nodes[-1].end_lineno if non_method_nodes else start)

#     return {
#         "chunk_id": f"{file_name}_{node.name}",
#         "source_file": file_stem,
#         "chunk_type": "class",
#         "parent_class": None,
#         "docstring": docstring,
#         "start_line": start,
#         "end_line": end,
#         "text": get_text(start, end)
#     }

In [10]:
def class_chunk(node, file_path):
    start = get_start_line(node)
    docstring = ast.get_docstring(node)

    # non-method body: everything in node.body that ISN'T a method
    non_method_nodes = [n for n in node.body if not isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]

    if any(isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) for n in node.body):
        end = (non_method_nodes[-1].end_lineno if non_method_nodes else start)
    else:
        end = node.end_lineno

    file_stem = file_path.stem if isinstance(file_path, Path) else str(file_path)

    return {
        "chunk_id": f"{file_stem}_{node.name}",
        "source_file": file_stem,
        "chunk_type": "class",
        "parent_class": None,
        "docstring": docstring,
        "start_line": start,
        "end_line": end,
        "text": get_text(start, end)
    }

In [11]:
results = class_chunk(node=tree.body[6],file_path = file)

print(results)

{'chunk_id': 'exceptions_RequestException', 'source_file': 'exceptions', 'chunk_type': 'class', 'parent_class': None, 'docstring': 'There was an ambiguous exception that occurred while handling your\nrequest.', 'start_line': 20, 'end_line': 26, 'text': 'class RequestException(IOError):\n    """There was an ambiguous exception that occurred while handling your\n    request.\n    """\n\n    response: Response | None\n    request: Request | PreparedRequest | None'}


In [12]:
test_src = '''
class Foo:
    """doc"""
    def bar(self):
        return 1
    baz = 2
'''
test_tree = ast.parse(test_src)
result = class_chunk(node=test_tree.body[0], file_path="test")
print(result["text"])

requests.exceptions
~~~~~~~~~~~~~~~~~~~

This module contains the set of Requests' exceptions.
"""


In [14]:
for node in tree.body:
    print(type(node).__name__, getattr(node, "name", None), getattr(node, "lineno", None), getattr(node, "end_lineno", None))

Expr None 1 7
ImportFrom None 9 9
Import None 11 11
Import None 12 12
Import None 13 13
Import None 14 14
ImportFrom None 15 15
ImportFrom None 17 26
ImportFrom None 27 27
ImportFrom None 28 28
ImportFrom None 29 29
ImportFrom None 30 30
ImportFrom None 31 31
ImportFrom None 32 32
ImportFrom None 33 33
ImportFrom None 34 34
ImportFrom None 36 36
ImportFrom None 37 37
ImportFrom None 38 38
ImportFrom None 39 50
ImportFrom None 51 51
ImportFrom None 52 52
ImportFrom None 53 60
Try None 62 67
If None 70 75
ImportFrom None 77 77
Assign None 79 79
Assign None 80 80
Assign None 81 81
Assign None 82 82
FunctionDef _urllib3_request_context 85 119
ClassDef BaseAdapter 122 155
ClassDef HTTPAdapter 158 748


### Final

In [ ]:
with open(file, encoding="utf-8") as f:
    content= f.read()

tree = ast.parse(content)
print(ast.dump(tree, indent=2))

In [56]:
def load_file(file):
    with open(file, encoding="utf-8") as f:
        content = f.read()   
    return content

def create_tree(load_file):
    file = load_file
    tree = ast.parse(file)
    return tree


def get_node_types(tree):
    results = []
    for node in tree.body:
        results.append((type(node).__name__, getattr(node, "name", None), getattr(node, "lineno", None), getattr(node, "end_lineno", None)))
    
    return results

In [96]:
def get_start_line(node):
    """Return the starting line of a node including the decorators"""
    if hasattr(node, "decorator_list") and node.decorator_list: #attribute and decorator check
        return node.decorator_list[0].lineno #The first decorator of the node
    return node.lineno

def get_text(start_line, end_line, content_lines):
    return "\n". join(content_lines[start_line - 1 : end_line])

In [57]:
raw = load_file(Path("exceptions.py"))
tree = create_tree(raw)
content_lines = raw.splitlines()
node_types = get_node_types(tree)

In [70]:
for node in tree.body[:5]:
    start = get_start_line(node)
    end = node.end_lineno
    print(f"Type: {type(node).__name__}, Attrs: {getattr(node, 'name', None)}, Start line: {start}, End line {end}")
    print(get_text(start, end))
    print("-" * 80)

Type: Expr, Attrs: None, Start line: 1, End line 6
"""
requests.exceptions
~~~~~~~~~~~~~~~~~~~

This module contains the set of Requests' exceptions.
"""
--------------------------------------------------------------------------------
Type: ImportFrom, Attrs: None, Start line: 8, End line 8
from __future__ import annotations
--------------------------------------------------------------------------------
Type: ImportFrom, Attrs: None, Start line: 10, End line 10
from typing import TYPE_CHECKING, Any
--------------------------------------------------------------------------------
Type: ImportFrom, Attrs: None, Start line: 12, End line 12
from urllib3.exceptions import HTTPError as BaseHTTPError
--------------------------------------------------------------------------------
Type: ImportFrom, Attrs: None, Start line: 14, End line 14
from .compat import JSONDecodeError as CompatJSONDecodeError
--------------------------------------------------------------------------------


In [71]:
node = tree.body[8]
print(get_text(get_start_line(node), node.end_lineno))

class JSONDecodeError(InvalidJSONError, CompatJSONDecodeError):
    """Couldn't decode the text into json"""

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        """
        Construct the JSONDecodeError instance first with all
        args. Then use it's args to construct the IOError so that
        the json specific args aren't used as IOError specific args
        and the error message from JSONDecodeError is preserved.
        """
        CompatJSONDecodeError.__init__(self, *args)
        InvalidJSONError.__init__(self, *self.args, **kwargs)

    def __reduce__(self) -> tuple[Any, ...] | str:
        """
        The __reduce__ method called when pickling the object must
        be the one from the JSONDecodeError (be it json/simplejson)
        as it expects all the arguments for instantiation, not just
        one like the IOError, and the MRO would by default call the
        __reduce__ method from the IOError due to the inheritance order.
        """
        retu

In [97]:
def build_class_chunk(node, file_path):
    file_stem = Path(file_path).stem
    start = get_start_line(node)
    docstring = ast.get_docstring(node)

    non_method_nodes = [n for n in node.body if not isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]

    # class signature + each non-method node's text, joined
    signature_line = content_lines[start-1] # the class X(Y): line
    body_pieces = [get_text(n.lineno, n.end_lineno, content_lines) for n in non_method_nodes]
    text = "\n".join([signature_line] + body_pieces)

    # end_line still needs a value for metadata. last non-method node, or signature if body is empty
    end = non_method_nodes[-1].end_lineno if non_method_nodes else start 

    chunks = {
        "chunk_id": f"{file_stem}_{node.name}",
        "source_file": file_stem,
        "chunk_type": "class",
        "parent_class": None,
        "docstring": docstring,
        "start_line": start,
        "end_line": end,
        "text": text
    }
    return chunks

In [73]:
test_src = '''
class Foo:
    """doc"""
    def bar(self):
        return 1
    baz = 2
'''
test_tree = ast.parse(test_src)
result = build_class_chunk(node=test_tree.body[0], file_path="test")
print(result["text"])

requests.exceptions
~~~~~~~~~~~~~~~~~~~
"""


- **the actual text(parsed) gives/shows signature + non-method attrs only ie start 2, end should be 4 instead of 8**
- **But the metadata claims that the lines span to the last *non-method* node's line**

In [78]:
new_test_src = '''
class Foo:
    """A class with mixed methods and attributess""" 
    def method1(self):
        return 1
    def method2(self):
        return 2 
    class_var = 52
'''

content_lines = new_test_src.splitlines()
test_tree = ast.parse(new_test_src)
result = build_class_chunk(node=test_tree.body[0], file_path="new_test")

print("Metadata:")
print(f"  start_line: {result['start_line']}")
print(f"  end_line: {result['end_line']}")
print("\nActual text:")
print(result['text'])
print("\n---")
print(f"Claims to span lines {result['start_line']}-{result['end_line']}")

Metadata:
  start_line: 2
  end_line: 8

Actual text:
class Foo:
    """A class with mixed methods and attributess""" 
    class_var = 52

---
Claims to span lines 2-8


In [108]:
def build_method_chunk(node, file_path, parent_class):
    file_stem = Path(file_path).stem
    start = get_start_line(node)
    end = node.end_lineno
    docstring = ast.get_docstring(node)

    chunks = {
        "chunk_id": f"{file_stem}_{parent_class}_{node.name}",
        "source_file": file_stem,
        "chunk_type": "method",
        "parent_class": parent_class,
        "docstring": docstring,
        "start_line": start,
        "end_line": end,
        "text": get_text(start, end, content_lines)
    }

    return chunks

def build_function_chunk(node, file_path):
    file_stem = Path(file_path)
    start = get_start_line(node)
    end = node.end_lineno
    docstring = ast.get_docstring(node)

    chunks = {
        "chunk_id": f"{file_stem}_{node.name}",
        "source_file": file_stem,
        "chunk_type": "function",
        "parent_class": None,
        "docstring": docstring,
        "start_line": start, 
        "end_line": end,
        "text": get_text(start, end, content_lines)
    }

    return chunks

In [112]:
sessions = load_file(Path("models.py"))
sessions_tree = create_tree(sessions)
content_lines = sessions.splitlines()
file = Path("models.py")

In [113]:
all_method_chunks = []
for node in sessions_tree.body:
    if isinstance(node, ast.ClassDef):
        parent_class = node.name
        for method_node in node.body:
            if isinstance(method_node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                chunk = build_method_chunk(node=method_node, file_path=file, parent_class=parent_class)
                all_method_chunks.append(chunk)


In [114]:
all_function_chunks = []
for node in sessions_tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        chunks = build_function_chunk(node=node, file_path=file)
        all_function_chunks.append(chunks)

In [115]:
print(f"Methods Found: {len(all_method_chunks)}")
for chunk in all_method_chunks[:3]:
    print(f"    {chunk}")
    print()

Methods Found: 51
    {'chunk_id': 'models_RequestEncodingMixin_path_url', 'source_file': 'models', 'chunk_type': 'method', 'parent_class': 'RequestEncodingMixin', 'docstring': 'Build the path URL to use.', 'start_line': 111, 'end_line': 130, 'text': '    @property\n    def path_url(self) -> str:\n        """Build the path URL to use."""\n\n        url: list[str] = []\n\n        p = urlsplit(cast(str, self.url))\n\n        path = p.path\n        if not path:\n            path = "/"\n\n        url.append(path)\n\n        query = p.query\n        if query:\n            url.append("?")\n            url.append(query)\n\n        return "".join(url)'}

    {'chunk_id': 'models_RequestEncodingMixin__encode_params', 'source_file': 'models', 'chunk_type': 'method', 'parent_class': 'RequestEncodingMixin', 'docstring': None, 'start_line': 132, 'end_line': 134, 'text': '    @overload\n    @staticmethod\n    def _encode_params(data: str) -> str: ...'}

    {'chunk_id': 'models_RequestEncodingMixin_

In [116]:
print(f"Functions found: {len(all_function_chunks)}")
for chunk in all_function_chunks:
    print(f" {chunk}")
    print()

Functions found: 0


In [131]:
def has_overload_decorator(node):
    """Check if a function is decorated with @overload (handles both Name and Attribute forms)."""
    if not hasattr(node, "decorator_list"):
        return False 
    
    for decorator in node.decorator_list:
        # Hanle @overload (name node)
        if isinstance(decorator, ast.Name) and decorator.id == "overload":
            return True
        # Hanle @overload (attr node)
        if isinstance(decorator, ast.Attribute) and decorator.attr == "overload":
            return True
    return False 

def filter_overload_stubs(nodes):
    """Filter out @overload stubs but keep the real implementation.
       Returns only the actual implementation functions."""
    # Group by function name
    functions_by_name = {}
    for node in nodes:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if node.name not in functions_by_name:
                functions_by_name[node.name] = []
            functions_by_name[node.name].append(node)
    # For each name, keep only non-overload versions or warn if non exist
    result = []
    for name, funcs in functions_by_name.items():
        real_imps = [f for f in funcs if not has_overload_decorator(f)]

        if real_imps:
            result.extend(real_imps)
        else:
            print(f" {name} has only @overload stubs, no implementation found")
    
    return result

In [139]:
sessions = load_file(Path("models.py"))
sessions_tree = create_tree(sessions)
content_lines = sessions.splitlines()
file = Path("models.py")

In [140]:
filtered_functions = filter_overload_stubs([n for n in sessions_tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))])

print(f"Fileterd functions(overload stubs removed): {len(filtered_functions)}")
for func in filtered_functions:
    print(f" {func.name}")

Fileterd functions(overload stubs removed): 0


In [141]:
all_method_chunks = []
for node in sessions_tree.body:
    if isinstance(node, ast.ClassDef):
        parent_class = node.name
        
        # Group methods by name within this class
        methods_by_name = {}
        for method_node in node.body:
            if isinstance(method_node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                if method_node.name not in methods_by_name:
                    methods_by_name[method_node.name] = []
                methods_by_name[method_node.name].append(method_node)
        
        # Filter and keep only real implementations
        for method_name, methods in methods_by_name.items():
            real_impls = [m for m in methods if not has_overload_decorator(m)]
            
            if real_impls:
                # Keep real implementations
                for method_node in real_impls:
                    chunk = build_method_chunk(node=method_node, file_path=file, parent_class=parent_class)
                    all_method_chunks.append(chunk)
            else:
                # Only stubs — keep the last one
                print(f"⚠️  {parent_class}.{method_name}: only @overload stubs, keeping last stub")
                method_node = methods[-1]
                chunk = build_method_chunk(node=method_node, file_path=file, parent_class=parent_class)
                all_method_chunks.append(chunk)

print(f"Methods found (stubs filtered): {len(all_method_chunks)}")
for chunk in all_method_chunks[:5]:
    print(f"  {chunk['chunk_id']}")

Methods found (stubs filtered): 43
  models_RequestEncodingMixin_path_url
  models_RequestEncodingMixin__encode_params
  models_RequestEncodingMixin__encode_files
  models_RequestHooksMixin_register_hook
  models_RequestHooksMixin_deregister_hook


In [144]:
# Search all_method_chunks for _encode_params
encode_params_chunks = [c for c in all_method_chunks if "_encode_params" in c['chunk_id']]

print(f"\n_encode_params chunks found: {len(encode_params_chunks)}")
for chunk in encode_params_chunks:
    print(f"  {chunk['chunk_id']} (lines {chunk['start_line']}-{chunk['end_line']})")
    print(f"    Decorators: {chunk.get('decorators', 'N/A')}")


_encode_params chunks found: 1
  models_RequestEncodingMixin__encode_params (lines 150-180)
    Decorators: N/A


In [142]:
def build_module_chunk(tree, file_path, content_lines):
    """Build a chunk from module_level nodes(imports, constants, etc.)"""
    file_stem = Path(file_path).stem 

    # non-class, non-function nodes
    non_code_nodes = [n for n in tree.body if not isinstance(n, (ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef))]

    if not non_code_nodes:
        return None

    docstring = ast.get_docstring(tree) 

    pieces = [get_text(get_start_line(n), n.end_lineno, content_lines) for n in non_code_nodes]
    text = "\n".join(pieces)

        # Metadata: first and last node positions
    start = get_start_line(non_code_nodes[0])
    end = non_code_nodes[-1].end_lineno
    
    chunk = {
        "chunk_id": f"{file_stem}_module",
        "source_file": file_stem,
        "chunk_type": "module",
        "parent_class": None,
        "docstring": docstring,
        "start_line": start,
        "end_line": end,
        "text": text,
        "is_sparse": len(non_code_nodes) > 1  # Flag if text has gaps
    }
    
    return chunk

In [143]:
module_chunk = build_module_chunk(sessions_tree, file, content_lines)
if module_chunk:
    print(f"Module chunk: {module_chunk['chunk_id']}")
    print(f"  Lines {module_chunk['start_line']}-{module_chunk['end_line']}")
    print(f"  Sparse: {module_chunk['is_sparse']}")

Module chunk: models_module
  Lines 1-105
  Sparse: True
